## Window functions — the pattern silver & gold lean on

A window function computes a value **per row** using a window of *other* rows around it — without collapsing the rows the way `groupBy` does. Three ingredients: `partitionBy` (the group), `orderBy` (the order within it), and optionally a frame (`rowsBetween` / `rangeBetween`).

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lag, sum as _sum, col

# 1. Deduplicate — rank newest-first, keep #1 (the SCD-latest pattern)
w = Window.partitionBy("customer_id").orderBy(col("transaction_at").desc())
df.withColumn("rn", row_number().over(w))

# 2. Period-over-period — the previous transaction's amount
df.withColumn("prev_amount", lag("amount").over(w))

# 3. Running total — an explicit frame
w_run = (Window.partitionBy("customer_id").orderBy("transaction_at")
               .rowsBetween(Window.unboundedPreceding, Window.currentRow))
df.withColumn("running_total", _sum("amount").over(w_run))
```

**The exam tests window functions for exactly three jobs:**

1. **Deduplication** — `row_number()`, keep `rn = 1` (deterministic, unlike `dropDuplicates`).
2. **Period-over-period** — `lag()` / `lead()` for "change since last transaction".
3. **Running totals** — `sum()` with a `rowsBetween` frame.

`rank()` and `dense_rank()` round out the ranking family — `rank` leaves gaps after ties, `dense_rank` does not.
